In [ ]:
# 12.09 sklearn version
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
# current_dir  = os.getcwd()
# project_root = os.path.abspath(os.path.join(current_dir, '..'))
project_root = "c:/big20/git/big20-ML-project2-team3/MercariPriceSuggestions"
# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd

from datetime import datetime
from typing import List, Dict, Any, Optional

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, StackingRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from hyperopt import hp
from scipy.sparse import hstack, csr_matrix

from utils.hyperopt_search import (
    hyperopt_search,
    train_and_evaluate_regressor,
)

import matplotlib.pyplot as plt
import seaborn           as sns
from sklearn.model_selection import cross_val_score


class MercariSklearnAnalyzer:
    """
    Mercari Price Suggestion - sklearn 기반 분석 클래스 (PyCaret 버전 기능을 계승한 버전)

    - 텍스트 + 카테고리/브랜드 + 숫자 피처 전처리
    - log1p(price)를 타깃으로 사용
    - LGBM/XGB/ExtraTrees + Stacking(Ridge meta)
    - hyperopt_search.py 유틸 (hyperopt_search, train_and_evaluate_regressor) 적극 활용
    - 모델/결과/캐시 저장:
        * ../models
        * ../results
        * ../images (시각화는 외부 util이 json을 읽어 생성한다고 가정)
    """

    # [__init__] start ##############################################################
    def __init__(
        self,
        random_state: int = 23,
        models_dir: str = "../models",
        results_dir: str = "../results",
        images_dir: str = "../images",
    ):
        """
        초기화 및 기본 경로 설정
        """
        self.random_state = random_state
        self.models_dir = models_dir
        self.results_dir = results_dir
        self.images_dir = images_dir

        self.train: Optional[pd.DataFrame] = None
        self.test: Optional[pd.DataFrame] = None

        self.vectorizer: Optional[TfidfVectorizer] = None

        # 학습/검증용
        self.X_train = None
        self.X_valid = None
        self.y_train = None
        self.y_valid = None

        # 전체 학습 데이터 (full train)
        self.X_train_full = None
        self.y_train_full = None

        # 캐글 제출용 test 피처
        self.X_test_kaggle = None

        # base_models: { name: { 'model_class', 'params', 'model', 'metrics', 'model_path', 'result_path' } }
        self.base_models: Dict[str, Dict[str, Any]] = {}

        # stacking 모델
        self.stack_model: Optional[Any] = None

        # best model
        self.best_model: Optional[Any] = None
        self.best_model_name: Optional[str] = None

        # 모든 모델 메트릭
        self.model_metrics: Dict[str, Dict[str, float]] = {}

        # 최종 best 모델 메트릭
        self.metrics: Dict[str, float] = {}
    # [__init__] end ================================================================


    # [load_data] start #############################################################
    def load_data(self, train_path: str, test_path: str, sep: str = "\t"):
        """
        Mercari 데이터 로딩 및 기본 정제

        - train: price > 0 필터 + 결측 처리
        - test: 결측 처리
        """
        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        # price > 0 필터 + NaN price 제거
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])

        # 결측 처리
        for df in [self.train, self.test]:
            df["brand_name"] = df["brand_name"].fillna("Unknown")
            df["category_name"] = df["category_name"].fillna("Unknown")
            df["item_description"] = df["item_description"].fillna("No description")

        print("✅ Data Loaded.")
        print(f"   train: {self.train.shape}, test: {self.test.shape}")
    # [load_data] end ===============================================================


    # [_split_category] start #######################################################
    def _split_category_columns(self, df: pd.DataFrame):
        """
        category_name을 'cat1', 'cat2', 'cat3'로 분리
        """
        cats = df["category_name"].str.split("/", n=2, expand=True)
        df["cat1"] = cats[0].fillna("NoCat1")
        df["cat2"] = cats[1].fillna("NoCat2") if cats.shape[1] > 1 else "NoCat2"
        df["cat3"] = cats[2].fillna("NoCat3") if cats.shape[1] > 2 else "NoCat3"
    # [_split_category] end =========================================================


    # [preprocess_all_staged] start ##################################################
    def preprocess_all_staged(
        self,
        use_cache: bool = True,
        save_cache: bool = True,
        debug: bool = True,
    ):
        """
        텍스트 + 카테고리/브랜드 + 숫자 피처 전처리 및 TF-IDF 벡터화

        - name + item_description → text_all
        - category_name 3분할 (cat1, cat2, cat3)
        - brand_name, cat1, cat2, cat3 빈도 인코딩
        - 숫자 피처:
            * item_condition_id
            * shipping
            * brand_freq
            * cat1_freq, cat2_freq, cat3_freq
        - 타깃: log1p(price)
        - 캐시: ../results/cache/mercari_preprocessed.pkl
        """
        if self.train is None or self.test is None:
            raise RuntimeError("load_data()를 먼저 호출하세요.")

        cache_dir = os.path.join(self.results_dir, "cache")
        os.makedirs(cache_dir, exist_ok=True)
        cache_path = os.path.join(cache_dir, "mercari_preprocessed.pkl")

        if use_cache and os.path.exists(cache_path):
            if debug:
                print(f"📦 캐시 로드: {cache_path}")
            with open(cache_path, "rb") as f:
                (
                    self.X_train,
                    self.X_valid,
                    self.y_train,
                    self.y_valid,
                    self.X_train_full,
                    self.y_train_full,
                    self.X_test_kaggle,
                    self.vectorizer,
                ) = pickle.load(f)
            return

        # category split
        self._split_category_columns(self.train)
        self._split_category_columns(self.test)

        # text_all 생성
        self.train["text_all"] = (
            self.train["name"].astype(str) + " " +
            self.train["item_description"].astype(str)
        )
        self.test["text_all"] = (
            self.test["name"].astype(str) + " " +
            self.test["item_description"].astype(str)
        )

        # target: log1p(price)
        y_log = np.log1p(self.train["price"].values)

        # 빈도 인코딩용 카운트
        brand_counts = self.train["brand_name"].value_counts()
        cat1_counts = self.train["cat1"].value_counts()
        cat2_counts = self.train["cat2"].value_counts()
        cat3_counts = self.train["cat3"].value_counts()

        # 빈도 인코딩
        self.train["brand_freq"] = self.train["brand_name"].map(brand_counts).fillna(0)
        self.test["brand_freq"] = self.test["brand_name"].map(brand_counts).fillna(0)

        self.train["cat1_freq"] = self.train["cat1"].map(cat1_counts).fillna(0)
        self.test["cat1_freq"] = self.test["cat1"].map(cat1_counts).fillna(0)

        self.train["cat2_freq"] = self.train["cat2"].map(cat2_counts).fillna(0)
        self.test["cat2_freq"] = self.test["cat2"].map(cat2_counts).fillna(0)

        self.train["cat3_freq"] = self.train["cat3"].map(cat3_counts).fillna(0)
        self.test["cat3_freq"] = self.test["cat3"].map(cat3_counts).fillna(0)

        # 숫자 피처
        num_cols = [
            "item_condition_id",
            "shipping",
            "brand_freq",
            "cat1_freq",
            "cat2_freq",
            "cat3_freq",
        ]

        num_train = self.train[num_cols].astype("float32").values
        num_test = self.test[num_cols].astype("float32").values

        # TF-IDF
        self.vectorizer = TfidfVectorizer(
            max_features=30000,
            stop_words="english",
        )

        X_text_train = self.vectorizer.fit_transform(self.train["text_all"])
        X_text_test = self.vectorizer.transform(self.test["text_all"])

        # sparse hstack
        X_full = hstack([X_text_train, csr_matrix(num_train)])
        X_test_kaggle = hstack([X_text_test, csr_matrix(num_test)])

        # train/valid split
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_full,
            y_log,
            test_size=0.2,
            random_state=self.random_state,
        )

        self.X_train = X_train
        self.X_valid = X_valid
        self.y_train = y_train
        self.y_valid = y_valid

        self.X_train_full = X_full
        self.y_train_full = y_log
        self.X_test_kaggle = X_test_kaggle

        if save_cache:
            with open(cache_path, "wb") as f:
                pickle.dump(
                    (
                        self.X_train,
                        self.X_valid,
                        self.y_train,
                        self.y_valid,
                        self.X_train_full,
                        self.y_train_full,
                        self.X_test_kaggle,
                        self.vectorizer,
                    ),
                    f,
                )
            if debug:
                print(f"💾 캐시 저장: {cache_path}")

        if debug:
            print("✅ Preprocessing done.")
    # [preprocess_all_staged] end ===================================================


    # [_train_single_model] start ###################################################
    def _train_single_model(
        self,
        model_name: str,
        model_class,
        search_space: Optional[dict] = None,
        max_evals: int = 50,
        use_hyperopt: bool = True,
    ):
        """
        단일 회귀 모델에 대해:
        - (옵션) hyperopt_search로 하이퍼파라미터 탐색
        - train_and_evaluate_regressor로 최종 학습 + 평가 + 저장
        - self.base_models / self.model_metrics 에 결과 반영
        """
        if self.X_train is None or self.y_train is None:
            raise RuntimeError("preprocess_all_staged()를 먼저 호출하세요.")

        print(f"\n🔹 [{model_name}] 회귀 모델 학습 시작")

        # 1) hyperopt로 best params 찾기 (log1p target 기준, RMSE(log) 최소화)
        if use_hyperopt and search_space is not None:
            result_search = hyperopt_search(
                model_class=model_class,
                search_space=search_space,
                X_train=self.X_train,
                y_train=self.y_train,
                scoring="neg_root_mean_squared_error",  # log space RMSE → RMSLE와 등가
                max_evals=max_evals,
                verbose=True,
            )
            best_params = result_search["best_params"]
        else:
            best_params = {"random_state": self.random_state}

        # 2) 최종 학습 + 평가 + 저장 (회귀용 유틸)
        result_train = train_and_evaluate_regressor(
            model_class=model_class,
            params=best_params,
            X_train=self.X_train,
            y_train=self.y_train,
            X_test=self.X_valid,
            y_test=self.y_valid,
            save_model=True,
            save_model_path=self.models_dir,
            save_result_path=self.results_dir,
            verbose=True,
            target_is_log1p=True,  # log1p(price) 타깃을 사용
        )

        model = result_train["model"]
        metrics = result_train["metrics"]
        model_path = result_train.get("model_path")
        result_path = result_train.get("result_path")

        print(f"  -> [{model_name}] RMSE: {metrics['rmse']:.4f}, RMSLE: {metrics['rmsle']:.4f}")

        # 3) 내부 저장
        self.base_models[model_name] = {
            "model_class": model_class,
            "params": best_params,
            "model": model,
            "metrics": metrics,
            "model_path": model_path,
            "result_path": result_path,
        }
        self.model_metrics[model_name] = metrics
    # [_train_single_model] end =====================================================


    # [train_base_models] start ######################################################
    def train_base_models(self, use_hyperopt: bool = True, max_evals: int = 50):
        """
        LGBMRegressor, XGBRegressor, ExtraTreesRegressor 3개 base 모델 학습
        """
        # hyperopt search space 예시 (필요 시 조정 가능)
        lgb_space = {
            "num_leaves": hp.quniform("num_leaves", 31, 255, 1),
            "learning_rate": hp.loguniform("learning_rate", -5, -1),
            "n_estimators": hp.quniform("n_estimators", 100, 1000, 50),
        }

        xgb_space = {
            "max_depth": hp.quniform("max_depth", 3, 12, 1),
            "learning_rate": hp.loguniform("learning_rate", -5, -1),
            "n_estimators": hp.quniform("n_estimators", 100, 1000, 50),
        }

        et_space = {
            "n_estimators": hp.quniform("n_estimators", 100, 500, 50),
            "max_depth": hp.quniform("max_depth", 5, 20, 1),
        }

        self._train_single_model(
            model_name="lgb",
            model_class=LGBMRegressor,
            search_space=lgb_space,
            max_evals=max_evals,
            use_hyperopt=use_hyperopt,
        )

        self._train_single_model(
            model_name="xgb",
            model_class=XGBRegressor,
            search_space=xgb_space,
            max_evals=max_evals,
            use_hyperopt=use_hyperopt,
        )

        self._train_single_model(
            model_name="et",
            model_class=ExtraTreesRegressor,
            search_space=et_space,
            max_evals=max_evals,
            use_hyperopt=use_hyperopt,
        )
    # [train_base_models] end ========================================================


    # [find_best_model] start ########################################################
    def find_best_model(self):
        """
        base 모델들 중 RMSLE가 가장 작은 모델을 best_model로 선택
        """
        if not self.base_models:
            raise RuntimeError("train_base_models()를 먼저 호출하세요.")

        best_name = None
        best_rmsle = float("inf")

        for name, info in self.base_models.items():
            rmsle = info["metrics"]["rmsle"]
            if rmsle < best_rmsle:
                best_rmsle = rmsle
                best_name = name

        self.best_model_name = best_name
        self.best_model = self.base_models[best_name]["model"]
        self.metrics = self.base_models[best_name]["metrics"]

        print(f"\n🏆 Best base model: {best_name}, RMSLE={best_rmsle:.4f}")
    # [find_best_model] end ==========================================================


    # [stack_models] start ###########################################################
    def stack_models(self):
        """
        base 모델(LGB, XGB, ET)을 이용하여 StackingRegressor (meta: Ridge) 생성

        - stacking 모델에 대해서도 회귀 메트릭 계산
        - base best_model 대비 좋으면 best_model을 stacking으로 교체
        """
        if not self.base_models:
            raise RuntimeError("train_base_models()를 먼저 호출하세요.")

        estimators = [
            (name, info["model"]) for name, info in self.base_models.items()
        ]

        stack = StackingRegressor(
            estimators=estimators,
            final_estimator=Ridge(alpha=1.0),
            passthrough=False,
            n_jobs=-1,
        )

        stack.fit(self.X_train, self.y_train)

        y_pred_log = stack.predict(self.X_valid)
        y_true_log = self.y_valid

        # 원래 price 스케일로 복원 후 메트릭 계산
        y_true = np.expm1(y_true_log)
        y_pred_price = np.maximum(np.expm1(y_pred_log), 0)

        rmse = float(np.sqrt(((y_true - y_pred_price) ** 2).mean()))
        mae = float(np.abs(y_true - y_pred_price).mean())
        r2 = float(np.corrcoef(y_true, y_pred_price)[0, 1] ** 2) if len(y_true) > 1 else 0.0
        rmsle = float(
            np.sqrt(
                np.mean((np.log1p(y_true) - np.log1p(y_pred_price)) ** 2)
            )
        )

        metrics = {
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "rmsle": rmsle,
        }

        self.stack_model = stack
        self.base_models["stacking"] = {
            "model_class": None,
            "params": {},
            "model": stack,
            "metrics": metrics,
            "model_path": None,
            "result_path": None,
        }
        self.model_metrics["stacking"] = metrics

        print(f"\n🔷 Stacking RMSE: {rmse:.4f}, RMSLE: {rmsle:.4f}")

        # 기존 best와 비교
        if self.best_model_name is None:
            self.best_model_name = "stacking"
            self.best_model = stack
            self.metrics = metrics
            print("  -> Stacking이 최초 best_model로 설정되었습니다.")
        else:
            best_rmsle = self.model_metrics[self.best_model_name]["rmsle"]
            if rmsle < best_rmsle:
                self.best_model_name = "stacking"
                self.best_model = stack
                self.metrics = metrics
                print("  -> Stacking이 기존 best_model보다 좋아서 교체되었습니다.")
    # [stack_models] end =============================================================


    # [evaluate] start ###############################################################
    def evaluate(self) -> Dict[str, float]:
        """
        현재 best_model 기준 메트릭 출력/리턴
        """
        if self.best_model_name is None or self.best_model is None:
            raise RuntimeError("best_model이 설정되지 않았습니다. find_best_model()/stack_models() 이후에 호출하세요.")

        metrics = self.model_metrics.get(self.best_model_name)
        if metrics is None:
            raise RuntimeError(f"model_metrics에 '{self.best_model_name}' 항목이 없습니다.")

        self.metrics = metrics

        print(f"\n📊 Final Evaluation (best_model = {self.best_model_name})")
        for k, v in metrics.items():
            print(f"  - {k}: {v:.4f}")

        return metrics
    # [evaluate] end ================================================================


    # [save_best] start ##############################################################
    def save_best(self) -> str:
        """
        best_model을 ../models 아래에 pickle로 저장
        """
        if self.best_model is None or self.best_model_name is None:
            raise RuntimeError("저장할 best_model이 없습니다.")

        os.makedirs(self.models_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"best_{self.best_model_name}_{timestamp}.pkl"
        save_path = os.path.join(self.models_dir, filename)

        with open(save_path, "wb") as f:
            pickle.dump(self.best_model, f)

        print(f"💾 Saved best model -> {save_path}")
        return save_path
    # [save_best] end ===============================================================


    # [save_all_metrics] start #######################################################
    def save_all_metrics(self) -> Dict[str, str]:
        """
        self.model_metrics 에 있는 모든 모델 메트릭을
        - 모델별 JSON
        - summary CSV
        형태로 ../results 아래에 저장

        시각화 util은 이 JSON/CSV를 사용해 ../images 아래에 그림을 저장한다고 가정.
        """
        if not self.model_metrics:
            raise RuntimeError("model_metrics가 비어 있습니다. train_base_models()/stack_models() 이후에 호출하세요.")

        os.makedirs(self.results_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        json_paths: Dict[str, str] = {}

        # 개별 JSON
        for model_name, metrics in self.model_metrics.items():
            filename = f"metrics_{model_name}_{timestamp}.json"
            save_path = os.path.join(self.results_dir, filename)

            payload = {
                "model_name": model_name,
                "metrics": metrics,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            }

            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(payload, f, ensure_ascii=False, indent=4)

            json_paths[model_name] = save_path

        # summary CSV
        rows = []
        for model_name, metrics in self.model_metrics.items():
            row = {"model_name": model_name}
            row.update(metrics)
            rows.append(row)

        summary_df = pd.DataFrame(rows)
        csv_filename = f"metrics_summary_{timestamp}.csv"
        csv_path = os.path.join(self.results_dir, csv_filename)
        summary_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        print("📁 Saved per-model metrics JSON files:")
        for m, p in json_paths.items():
            print(f"  - {m}: {p}")
        print(f"📊 Saved metrics summary CSV -> {csv_path}")

        return {"csv": csv_path, "results_dir": self.results_dir}
    # [save_all_metrics] end ========================================================


    # [predict] start ################################################################
    def predict(self, text_list: List[str]) -> np.ndarray:
        """
        새 텍스트 리스트에 대해 가격 예측

        - 입력: "name + description" 형식의 텍스트 리스트
        - 출력: price (원래 스케일, expm1 적용 완료)
        """
        if self.vectorizer is None:
            raise RuntimeError("vectorizer가 없습니다. preprocess_all_staged()를 먼저 호출하세요.")
        if self.best_model is None:
            raise RuntimeError("best_model이 없습니다. 학습 및 find_best_model()/stack_models() 이후에 호출하세요.")

        # 텍스트만 받았다고 가정 → numeric 피처는 0으로 채운다는 단순 버전
        X_text = self.vectorizer.transform(text_list)
        num_dummy = np.zeros((len(text_list), 6), dtype="float32")  # [item_cond, shipping, brand_freq, cat1, cat2, cat3]
        X_new = hstack([X_text, csr_matrix(num_dummy)])

        y_pred_log = self.best_model.predict(X_new)
        y_pred_price = np.maximum(np.expm1(y_pred_log), 0)

        return y_pred_price
    # [predict] end ==================================================================


    # [predict_test_and_save_submission] start #######################################
    def predict_test_and_save_submission(self, filename_prefix: str = "submission") -> str:
        """
        캐글 test 데이터 전체에 대해 best_model로 예측하고
        ../results 아래에 submission CSV 저장

        - 컬럼: ['test_id', 'price']
          (test에 'test_id' 컬럼이 있다고 가정. 없으면 단순 index 사용)
        """
        if self.best_model is None:
            raise RuntimeError("best_model이 없습니다.")
        if self.X_test_kaggle is None or self.test is None:
            raise RuntimeError("preprocess_all_staged()가 실행되지 않았거나 test 데이터가 없습니다.")

        y_pred_log = self.best_model.predict(self.X_test_kaggle)
        y_pred_price = np.maximum(np.expm1(y_pred_log), 0)

        # test_id 컬럼이 있으면 사용, 없으면 인덱스 사용
        if "test_id" in self.test.columns:
            ids = self.test["test_id"].values
        elif "id" in self.test.columns:
            ids = self.test["id"].values
        else:
            ids = np.arange(len(self.test))

        sub_df = pd.DataFrame({"test_id": ids, "price": y_pred_price})

        os.makedirs(self.results_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{filename_prefix}_{self.best_model_name}_{timestamp}.csv"
        save_path = os.path.join(self.results_dir, filename)

        sub_df.to_csv(save_path, index=False, encoding="utf-8-sig")
        print(f"📄 Saved submission CSV -> {save_path}")

        return save_path
    # [predict_test_and_save_submission] end ========================================




    # cross_validate_best start ###########################
    def cross_validate_best(self, cv: int = 5, scoring: str = "neg_root_mean_squared_error"):
        """
        Best 모델에 대해 K-Fold 교차 검증 수행 및 결과 시각화
        
        단일 train/valid split으로 평가한 성능이 과적합되지 않았는지 확인하기 위해
        교차 검증을 통해 모델의 안정성과 일반화 성능을 평가합니다.
        
        Parameters:
        -----------
        cv : int, default=5
            교차 검증 fold 수
        scoring : str, default='neg_root_mean_squared_error'
            평가 지표 (회귀용: 'neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2')
        
        Returns:
        --------
        dict : {
            'cv_scores': fold별 점수 리스트,
            'mean_score': 평균 점수,
            'std_score': 표준편차,
            'cv_rmsle': RMSLE 평균 (log1p 타겟 기준)
        }
        
        Examples:
        ---------
        >>> cv_result = analyzer.cross_validate_best(cv=5)
        >>> print(f"CV RMSLE: {cv_result['cv_rmsle']:.4f} (+/- {cv_result['std_score']:.4f})")
        
        Notes:
        ------
        - X_train_full과 y_train_full (전체 학습 데이터)을 사용하여 교차 검증
        - 결과 시각화를 ../images/cv_results_{timestamp}.png로 저장
        - scoring이 'neg_'로 시작하는 경우 음수 부호 제거하여 출력
        """
        if self.best_model is None or self.best_model_name is None:
            raise RuntimeError("best_model이 없습니다. find_best_model()/stack_models() 이후에 호출하세요.")
        
        if self.X_train_full is None or self.y_train_full is None:
            raise RuntimeError("X_train_full이 없습니다. preprocess_all_staged() 이후에 호출하세요.")
        
        print(f"\n{'='*80}")
        print(f"  {self.best_model_name} 모델 {cv}-Fold 교차 검증")
        print(f"{'='*80}")
        
        # 교차 검증 수행
        cv_scores = cross_val_score(
            self.best_model,
            self.X_train_full,
            self.y_train_full,
            cv=cv,
            scoring=scoring,
            n_jobs=-1
        )
        
        # 음수 부호 제거 (neg_root_mean_squared_error 등)
        if scoring.startswith('neg_'):
            cv_scores = -cv_scores
            metric_name = scoring.replace('neg_', '').upper()
        else:
            metric_name = scoring.upper()
        
        mean_score = cv_scores.mean()
        std_score = cv_scores.std()
        
        # RMSLE 계산 (log1p 타겟이므로 RMSE(log) = RMSLE)
        cv_rmsle = mean_score if 'root_mean_squared' in scoring else None
        
        print(f"\n교차 검증 결과 ({metric_name}):")
        print(f"  - Fold별 점수: {cv_scores}")
        print(f"  - 평균: {mean_score:.4f}")
        print(f"  - 표준편차: {std_score:.4f}")
        if cv_rmsle:
            print(f"  - CV RMSLE: {cv_rmsle:.4f} (+/- {std_score:.4f})")
        
        # 시각화
        os.makedirs(self.images_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Bar plot
        folds = [f'Fold {i+1}' for i in range(cv)]
        colors = ['#3498db' if score < mean_score else '#e74c3c' for score in cv_scores]
        bars = ax.bar(folds, cv_scores, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
        
        # 평균선
        ax.axhline(mean_score, color='green', linestyle='--', linewidth=2, label=f'Mean: {mean_score:.4f}')
        ax.axhline(mean_score + std_score, color='orange', linestyle=':', linewidth=1.5, label=f'+1 Std: {mean_score+std_score:.4f}')
        ax.axhline(mean_score - std_score, color='orange', linestyle=':', linewidth=1.5, label=f'-1 Std: {mean_score-std_score:.4f}')
        
        # 그래프 꾸미기
        ax.set_xlabel('Fold', fontsize=12, fontweight='bold')
        ax.set_ylabel(f'{metric_name} Score', fontsize=12, fontweight='bold')
        ax.set_title(f'{self.best_model_name} - {cv}-Fold Cross Validation\nMean: {mean_score:.4f} (+/- {std_score:.4f})', 
                    fontsize=14, fontweight='bold', pad=20)
        ax.legend(loc='upper right', fontsize=10)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        
        # 값 표시
        for bar, score in zip(bars, cv_scores):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{score:.4f}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        
        save_path = os.path.join(self.images_dir, f'cv_results_{self.best_model_name}_{timestamp}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n📊 시각화 저장: {save_path}")
        plt.close()
        
        return {
            'cv_scores': cv_scores.tolist(),
            'mean_score': float(mean_score),
            'std_score': float(std_score),
            'cv_rmsle': float(cv_rmsle) if cv_rmsle else None
        }
    # cross_validate_best end ======================================


    # plot_feature_importance start ###########################
    def plot_feature_importance(self, top_n: int = 20, importance_type: str = 'gain'):
        """
        Best 모델의 피처 중요도 시각화 (트리 기반 모델 전용)
        
        LGBM, XGB, ExtraTrees 등 트리 기반 모델의 feature importance를 시각화합니다.
        TF-IDF 단어와 수치형 피처의 중요도를 함께 확인할 수 있습니다.
        
        Parameters:
        -----------
        top_n : int, default=20
            상위 N개 중요 피처만 시각화
        importance_type : str, default='gain'
            중요도 계산 방식 (LGBM/XGB: 'gain', 'split', 'cover' 등)
            - 'gain': 정보 이득 (가장 일반적)
            - 'split': 분할 횟수
            - 'weight': sklearn의 feature_importances_ 사용
        
        Returns:
        --------
        pd.DataFrame : 피처명과 중요도를 담은 데이터프레임 (상위 top_n개)
        
        Examples:
        ---------
        >>> # 상위 30개 피처 확인
        >>> importance_df = analyzer.plot_feature_importance(top_n=30)
        >>> print(importance_df.head(10))
        
        Notes:
        ------
        - Stacking 모델은 피처 중요도를 제공하지 않으므로 base 모델 사용 권장
        - TF-IDF 피처명은 vectorizer.get_feature_names_out()로 추출
        - 수치형 피처: item_condition_id, shipping, brand_freq, cat1/2/3_freq
        - 결과를 ../images/feature_importance_{timestamp}.png로 저장
        """
        if self.best_model is None or self.best_model_name is None:
            raise RuntimeError("best_model이 없습니다.")
        
        if self.vectorizer is None:
            raise RuntimeError("vectorizer가 없습니다. preprocess_all_staged() 이후에 호출하세요.")
        
        print(f"\n{'='*80}")
        print(f"  {self.best_model_name} 모델 피처 중요도 분석")
        print(f"{'='*80}")
        
        # 피처 이름 생성
        tfidf_features = self.vectorizer.get_feature_names_out().tolist()
        numeric_features = [
            'item_condition_id', 'shipping', 'brand_freq',
            'cat1_freq', 'cat2_freq', 'cat3_freq'
        ]
        all_features = tfidf_features + numeric_features
        
        # 피처 중요도 추출
        try:
            model = self.best_model
            
            # Stacking 모델인 경우 첫 번째 base estimator 사용
            if self.best_model_name == 'stacking':
                print("⚠️ Stacking 모델은 피처 중요도를 직접 제공하지 않습니다.")
                print("   첫 번째 base estimator의 중요도를 사용합니다.")
                model = self.best_model.estimators_[0]
            
            # LGBM/XGB: booster.feature_importance() 사용
            if hasattr(model, 'booster_'):
                importances = model.booster_.feature_importance(importance_type=importance_type)
            # sklearn의 feature_importances_ 사용
            elif hasattr(model, 'feature_importances_'):
                importances = model.feature_importances_
            else:
                raise AttributeError(f"{self.best_model_name} 모델은 피처 중요도를 제공하지 않습니다.")
            
        except Exception as e:
            print(f"❌ 피처 중요도 추출 실패: {e}")
            return None
        
        # 데이터프레임 생성
        importance_df = pd.DataFrame({
            'feature': all_features,
            'importance': importances
        })
        
        # 중요도 기준 정렬 및 상위 N개 선택
        importance_df = importance_df.sort_values('importance', ascending=False).head(top_n)
        
        print(f"\n상위 {top_n}개 중요 피처:")
        print(importance_df.to_string(index=False))
        
        # 시각화
        os.makedirs(self.images_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Horizontal bar plot
        colors = sns.color_palette('viridis', len(importance_df))
        bars = ax.barh(importance_df['feature'], importance_df['importance'], color=colors, edgecolor='black', linewidth=0.8)
        
        # 그래프 꾸미기
        ax.set_xlabel('Importance Score', fontsize=12, fontweight='bold')
        ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
        ax.set_title(f'{self.best_model_name} - Top {top_n} Feature Importance\n({importance_type.capitalize()} method)', 
                    fontsize=14, fontweight='bold', pad=20)
        ax.invert_yaxis()  # 상위 피처가 위로 오도록
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        
        # 값 표시
        for bar, value in zip(bars, importance_df['importance']):
            width = bar.get_width()
            ax.text(width, bar.get_y() + bar.get_height()/2.,
                    f' {value:.1f}',
                    ha='left', va='center', fontsize=9, fontweight='bold')
        
        plt.tight_layout()
        
        save_path = os.path.join(self.images_dir, f'feature_importance_{self.best_model_name}_{timestamp}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n📊 시각화 저장: {save_path}")
        plt.close()
        
        return importance_df
    # plot_feature_importance end ======================================


    # residual_analysis start ###########################
    def residual_analysis(self, save_samples: bool = True, n_samples: int = 20):
        """
        Best 모델의 잔차(Residual) 분석 및 시각화
        
        예측값과 실제값의 차이(잔차)를 분석하여 모델의 예측 패턴을 파악합니다.
        과대/과소 예측 경향, 가격대별 오차 분포 등을 확인할 수 있습니다.
        
        Parameters:
        -----------
        save_samples : bool, default=True
            최대/최소 오차 샘플을 CSV로 저장할지 여부
        n_samples : int, default=20
            저장할 샘플 개수 (상위/하위 각각)
        
        Returns:
        --------
        dict : {
            'residuals': 잔차 배열,
            'mean_residual': 평균 잔차,
            'std_residual': 잔차 표준편차,
            'max_overpredict': 최대 과대예측 인덱스,
            'max_underpredict': 최대 과소예측 인덱스
        }
        
        Examples:
        ---------
        >>> residual_info = analyzer.residual_analysis(save_samples=True, n_samples=50)
        >>> print(f"평균 잔차: {residual_info['mean_residual']:.2f}")
        
        Notes:
        ------
        - Residual = Predicted Price - Actual Price
        - 양수: 과대예측 (모델이 실제보다 높게 예측)
        - 음수: 과소예측 (모델이 실제보다 낮게 예측)
        - 4개 시각화 생성:
            1. Residual vs Predicted (잔차 분산 확인)
            2. Residual Distribution (정규분포 확인)
            3. Actual vs Predicted (45도선 확인)
            4. Price Range Error (가격대별 오차)
        - 결과를 ../images/residual_analysis_{timestamp}.png로 저장
        - 최대 오차 샘플을 ../results/residual_samples_{timestamp}.csv로 저장
        """
        if self.best_model is None or self.best_model_name is None:
            raise RuntimeError("best_model이 없습니다.")
        
        if self.X_valid is None or self.y_valid is None:
            raise RuntimeError("검증 데이터가 없습니다. preprocess_all_staged() 이후에 호출하세요.")
        
        print(f"\n{'='*80}")
        print(f"  {self.best_model_name} 모델 잔차 분석")
        print(f"{'='*80}")
        
        # 예측
        y_pred_log = self.best_model.predict(self.X_valid)
        y_true_log = self.y_valid
        
        # 원래 스케일로 복원
        y_true = np.expm1(y_true_log)
        y_pred = np.maximum(np.expm1(y_pred_log), 0)
        
        # 잔차 계산 (Predicted - Actual)
        residuals = y_pred - y_true
        
        mean_residual = residuals.mean()
        std_residual = residuals.std()
        
        print(f"\n잔차 통계:")
        print(f"  - 평균 잔차: ${mean_residual:.2f}")
        print(f"  - 표준편차: ${std_residual:.2f}")
        print(f"  - 최대 과대예측: ${residuals.max():.2f}")
        print(f"  - 최대 과소예측: ${residuals.min():.2f}")
        
        # 최대 오차 인덱스
        max_overpredict_idx = residuals.argmax()
        max_underpredict_idx = residuals.argmin()
        
        print(f"\n극단 케이스:")
        print(f"  - 최대 과대예측: 실제=${y_true[max_overpredict_idx]:.2f}, 예측=${y_pred[max_overpredict_idx]:.2f}")
        print(f"  - 최대 과소예측: 실제=${y_true[max_underpredict_idx]:.2f}, 예측=${y_pred[max_underpredict_idx]:.2f}")
        
        # 샘플 저장
        if save_samples:
            os.makedirs(self.results_dir, exist_ok=True)
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            
            # 상위/하위 N개씩 추출
            sorted_indices = np.argsort(residuals)
            worst_underpredict = sorted_indices[:n_samples]  # 가장 낮게 예측
            worst_overpredict = sorted_indices[-n_samples:]  # 가장 높게 예측
            
            sample_indices = np.concatenate([worst_underpredict, worst_overpredict])
            
            samples_df = pd.DataFrame({
                'index': sample_indices,
                'actual_price': y_true[sample_indices],
                'predicted_price': y_pred[sample_indices],
                'residual': residuals[sample_indices],
                'abs_error': np.abs(residuals[sample_indices]),
                'error_type': ['underpredict'] * n_samples + ['overpredict'] * n_samples
            })
            
            csv_path = os.path.join(self.results_dir, f'residual_samples_{self.best_model_name}_{timestamp}.csv')
            samples_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
            print(f"\n💾 샘플 저장: {csv_path}")
        
        # 시각화 (2x2 subplot)
        os.makedirs(self.images_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f'{self.best_model_name} - Residual Analysis', fontsize=16, fontweight='bold', y=0.995)
        
        # 1. Residual vs Predicted
        ax1 = axes[0, 0]
        ax1.scatter(y_pred, residuals, alpha=0.3, s=10, c='steelblue', edgecolors='none')
        ax1.axhline(0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction')
        ax1.set_xlabel('Predicted Price ($)', fontsize=11, fontweight='bold')
        ax1.set_ylabel('Residual ($)', fontsize=11, fontweight='bold')
        ax1.set_title('Residual vs Predicted Price', fontsize=12, fontweight='bold', pad=10)
        ax1.legend()
        ax1.grid(alpha=0.3, linestyle='--')
        
        # 2. Residual Distribution
        ax2 = axes[0, 1]
        ax2.hist(residuals, bins=50, color='coral', alpha=0.7, edgecolor='black', linewidth=0.8)
        ax2.axvline(mean_residual, color='red', linestyle='--', linewidth=2, label=f'Mean: ${mean_residual:.2f}')
        ax2.axvline(0, color='green', linestyle='-', linewidth=2, label='Zero Residual')
        ax2.set_xlabel('Residual ($)', fontsize=11, fontweight='bold')
        ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
        ax2.set_title('Residual Distribution', fontsize=12, fontweight='bold', pad=10)
        ax2.legend()
        ax2.grid(axis='y', alpha=0.3, linestyle='--')
        
        # 3. Actual vs Predicted
        ax3 = axes[1, 0]
        ax3.scatter(y_true, y_pred, alpha=0.3, s=10, c='mediumseagreen', edgecolors='none')
        
        # 45도선 (완벽한 예측선)
        max_val = max(y_true.max(), y_pred.max())
        ax3.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')
        
        ax3.set_xlabel('Actual Price ($)', fontsize=11, fontweight='bold')
        ax3.set_ylabel('Predicted Price ($)', fontsize=11, fontweight='bold')
        ax3.set_title('Actual vs Predicted Price', fontsize=12, fontweight='bold', pad=10)
        ax3.legend()
        ax3.grid(alpha=0.3, linestyle='--')
        
        # 4. Price Range Error
        ax4 = axes[1, 1]
        
        # 가격대별 구간 나누기
        price_bins = [0, 10, 20, 50, 100, 200, 500, np.inf]
        bin_labels = ['0-10', '10-20', '20-50', '50-100', '100-200', '200-500', '500+']
        
        price_ranges = pd.cut(y_true, bins=price_bins, labels=bin_labels)
        
        mae_by_range = []
        for label in bin_labels:
            mask = (price_ranges == label)
            if mask.sum() > 0:
                mae = np.abs(residuals[mask]).mean()
                mae_by_range.append(mae)
            else:
                mae_by_range.append(0)
        
        colors_range = sns.color_palette('Reds', len(bin_labels))
        bars = ax4.bar(bin_labels, mae_by_range, color=colors_range, alpha=0.8, edgecolor='black', linewidth=1)
        
        ax4.set_xlabel('Price Range ($)', fontsize=11, fontweight='bold')
        ax4.set_ylabel('Mean Absolute Error ($)', fontsize=11, fontweight='bold')
        ax4.set_title('MAE by Price Range', fontsize=12, fontweight='bold', pad=10)
        ax4.grid(axis='y', alpha=0.3, linestyle='--')
        
        # 값 표시
        for bar, mae in zip(bars, mae_by_range):
            height = bar.get_height()
            if height > 0:
                ax4.text(bar.get_x() + bar.get_width()/2., height,
                        f'${mae:.1f}',
                        ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        plt.tight_layout()
        
        save_path = os.path.join(self.images_dir, f'residual_analysis_{self.best_model_name}_{timestamp}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n📊 시각화 저장: {save_path}")
        plt.close()
        
        return {
            'residuals': residuals,
            'mean_residual': float(mean_residual),
            'std_residual': float(std_residual),
            'max_overpredict': int(max_overpredict_idx),
            'max_underpredict': int(max_underpredict_idx)
        }
    # residual_analysis end ======================================


    # ============================================================================
    # 사용 예시 (MercariSklearnAnalyzer 인스턴스에서 호출)
    # ============================================================================
    """
    # 1. 교차 검증
    cv_result = analyzer.cross_validate_best(cv=5)
    print(f"CV RMSLE: {cv_result['cv_rmsle']:.4f} (+/- {cv_result['std_score']:.4f})")

    # 2. 피처 중요도 (상위 30개)
    importance_df = analyzer.plot_feature_importance(top_n=30, importance_type='gain')
    print(importance_df.head(10))

    # 3. 잔차 분석 (상위/하위 50개씩 샘플 저장)
    residual_info = analyzer.residual_analysis(save_samples=True, n_samples=50)
    print(f"평균 잔차: ${residual_info['mean_residual']:.2f}")
    print(f"잔차 표준편차: ${residual_info['std_residual']:.2f}")
    """
    
# End of Class     

In [ ]:
# ============================================================================
# Mercari Price Prediction - 최종 실행 Pipeline
# ============================================================================

import sys
import os
import warnings
warnings.filterwarnings('ignore')

# 프로젝트 루트 추가
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)




# ============================================================================
# STEP 0: 초기화
# ============================================================================
print("=" * 80)
print("  Mercari Price Prediction Pipeline")
print("=" * 80)

analyzer = MercariSklearnAnalyzer(
    random_state=23,
    models_dir="../models",
    results_dir="../results",
    images_dir="../images"
)


# ============================================================================
# STEP 1: 데이터 로딩
# ============================================================================
print("\n[STEP 1] 데이터 로딩")
print("-" * 80)

analyzer.load_data(
    train_path="../data/train.tsv",
    test_path="../data/test.tsv",
    sep="\t"
)


# ============================================================================
# STEP 2: 전처리 (TF-IDF + 피처 엔지니어링)
# ============================================================================
print("\n[STEP 2] 전처리 및 피처 생성")
print("-" * 80)

analyzer.preprocess_all_staged(
    use_cache=True,      # 캐시 사용 (2번째 실행부터 빠름)
    save_cache=True,     # 캐시 저장
    debug=True
)

print(f"""
전처리 완료:
  - X_train: {analyzer.X_train.shape}
  - X_valid: {analyzer.X_valid.shape}
  - X_test_kaggle: {analyzer.X_test_kaggle.shape}
  - Target: log1p(price)
""")


# ============================================================================
# STEP 3: Base 모델 학습 (LGBM, XGB, ExtraTrees)
# ============================================================================
print("\n[STEP 3] Base 모델 학습 (Hyperopt + 3개 모델)")
print("-" * 80)

# 옵션 1: Hyperopt 사용 (시간 소요, 성능 최적화)
analyzer.train_base_models(
    use_hyperopt=True,
    max_evals=50  # 빠른 테스트: 20, 실전: 100+
)

# 옵션 2: Hyperopt 생략 (빠른 테스트용)
# analyzer.train_base_models(use_hyperopt=False)


# ============================================================================
# STEP 4: Best Base 모델 선택
# ============================================================================
print("\n[STEP 4] Best Base 모델 선택")
print("-" * 80)

analyzer.find_best_model()


# ============================================================================
# STEP 5: Stacking 앙상블
# ============================================================================
print("\n[STEP 5] Stacking 앙상블 (LGBM+XGB+ET → Ridge)")
print("-" * 80)

analyzer.stack_models()


# ============================================================================
# STEP 6: 최종 평가
# ============================================================================
print("\n[STEP 6] 최종 평가")
print("-" * 80)

final_metrics = analyzer.evaluate()

print(f"""
최종 모델: {analyzer.best_model_name}
  - RMSLE: {final_metrics['rmsle']:.4f}
  - RMSE:  {final_metrics['rmse']:.2f}
  - MAE:   {final_metrics['mae']:.2f}
  - R²:    {final_metrics['r2']:.4f}
""")


# ============================================================================
# STEP 7: 모델 및 결과 저장
# ============================================================================
print("\n[STEP 7] 모델 및 결과 저장")
print("-" * 80)

# Best 모델 저장
best_model_path = analyzer.save_best()

# 모든 모델 메트릭 저장 (JSON + CSV)
metrics_paths = analyzer.save_all_metrics()

print(f"""
저장 완료:
  - Best 모델: {best_model_path}
  - 메트릭 CSV: {metrics_paths['csv']}
  - 개별 JSON: {metrics_paths['results_dir']}
""")


# ============================================================================
# STEP 8: 캐글 제출 파일 생성
# ============================================================================
print("\n[STEP 8] 캐글 제출 파일 생성")
print("-" * 80)

submission_path = analyzer.predict_test_and_save_submission(
    filename_prefix="mercari_submission"
)

print(f"📤 제출 파일: {submission_path}")


# ============================================================================
# STEP 9: 개별 예측 테스트 (선택 사항)
# ============================================================================
print("\n[STEP 9] 개별 예측 테스트 (샘플)")
print("-" * 80)

sample_texts = [
    "Nike Air Max Running Shoes Brand New in Box",
    "iPhone 13 Pro 256GB Unlocked",
    "Vintage Coach Leather Handbag"
]

predictions = analyzer.predict(sample_texts)

for text, price in zip(sample_texts, predictions):
    print(f"  - {text[:50]:<50} → ${price:.2f}")


# ============================================================================
# 완료
# ============================================================================
print("\n" + "=" * 80)
print("  Pipeline 완료!")
print("=" * 80)

print(f"""
다음 단계:
  1. 시각화: {metrics_paths['csv']} 파일로 성능 비교 차트 생성
  2. 캐글 제출: {submission_path} 업로드
  3. 모델 개선:
     - max_evals 증가 (50 → 100+)
     - TF-IDF max_features 조정 (30000 → 50000)
     - 추가 피처 엔지니어링 (description 길이, 단어 수 등)
""")


# ============================================================================
# [선택] 빠른 실행 모드 (디버깅용)
# ============================================================================
"""
# 5분 안에 전체 파이프라인 실행 (성능은 낮음)
analyzer = MercariSklearnAnalyzer(random_state=23)
analyzer.load_data("../data/train.tsv", "../data/test.tsv", sep="\t")
analyzer.preprocess_all_staged(use_cache=True, save_cache=True)

# Hyperopt 생략 버전
analyzer.train_base_models(use_hyperopt=False)
analyzer.find_best_model()
analyzer.stack_models()
analyzer.evaluate()

# 저장 및 제출
analyzer.save_best()
analyzer.save_all_metrics()
analyzer.predict_test_and_save_submission()
"""

# Pipeline 마지막에 추가
# ============================================================================
# STEP 10: 추가 분석 (선택)
# ============================================================================

# 1) 교차 검증으로 모델 안정성 확인
print("\n[분석 1] 교차 검증")
cv_result = analyzer.cross_validate_best(cv=5)

# 2) 중요한 피처 확인 (브랜드? 카테고리? 특정 단어?)
print("\n[분석 2] 피처 중요도")
importance_df = analyzer.plot_feature_importance(top_n=30)

# 3) 예측 오차 패턴 분석
print("\n[분석 3] 잔차 분석")
residual_info = analyzer.residual_analysis(save_samples=True, n_samples=50)

print(f"""
분석 완료!
  - CV RMSLE: {cv_result['cv_rmsle']:.4f} (+/- {cv_result['std_score']:.4f})
  - 상위 피처: {importance_df.iloc[0]['feature']}
  - 평균 잔차: ${residual_info['mean_residual']:.2f}
""")